# Chatbots

* ELIZA replied with pattern matches.
* DialogFlow mapped intents.
* GPT answered from weights.
* Claude runs tools and verifies

## Problem Definition

## Basic Concept

### Evolution

#### Rule-based

Hand-authored patterns match user input and produce responses.

#### Retrieval-base

FAQ-style. Encode every pair of (utterance, response). At runtime, encode the user's message and retrieve the nearest stored response.

#### Neural

Encoder-Decoder trained on conversation logs. Generate responses from scratch.

#### LLM agents

A language model wrapped in a loop that plans, calls tools, and verifies outcomes.



# Build your Own

## Rule based pattern matching

In [1]:
import re
import sys
from pathlib import Path

sys.path.insert(0, str(Path("../../00_Common").resolve()))
from user_tools import SectionPrinter

class RulePattern:
    def __init__(self, pattern, response_template):
        self.regex = re.compile(pattern, re.IGNORECASE)
        self.template = response_template

PATTERNS = [
    RulePattern(r"My name is (\w+)", "Hello, {0}.")
]

def rule_based_respond(user_input):
    for pattern in PATTERNS:
        m = pattern.regex.match(user_input.strip())
        if m:
            #print(*m.groups())
            return pattern.template.format(*m.groups())
    return "I'm sorry, I don't understand."

with SectionPrinter("Rule-Based"):
    print(rule_based_respond("My name is John"))

=========================Rule-Based=========================
Hello, John.


## Retrievel-based

In [2]:
from sentence_transformers import SentenceTransformer

import numpy as np

FAQ = [
    ("What is the capital of France?", "Paris"),
    ("What is the capital of Germany?", "Berlin"),
    ("What is the capital of Italy?", "Rome"),
    ("What is the capital of Spain?", "Madrid"),
    ("What is the capital of Portugal?", "Lisbon"),
    ("What is the capital of Greece?", "Athens"),
    ("What is the capital of Turkey?", "Ankara"),
]

encoder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

faq_questions = [q for q, _ in FAQ]
faq_embeddings = encoder.encode(faq_questions, normalize_embeddings=True)

def faq_respond(user_input, threshold):
    q_emb = encoder.encode([user_input], normalize_embeddings=True)[0]

    sim = faq_embeddings @ q_emb
    best = int(np.argmax(sim))
    if sim[best] < threshold:
        return "I'm sorry, I don't know the answer."
    return FAQ[best][1]

with SectionPrinter("Retrieval-Based"):
    print(faq_respond("What is the capital of France?", 0.5))
    print(faq_respond("What is the capital of Germany?", 0.5))
    print(faq_respond("What is the capital of Italy?", 0.5))
    print(faq_respond("What is the capital of Spain?", 0.5))
    print(faq_respond("What is the capital of Portugal?", 0.5))
    print(faq_respond("What is the capital of Greece?", 0.5))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

======================Retrieval-Based=======================
Paris
Berlin
Rome
Madrid
Lisbon
Athens


## Neural Generation

In [5]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "google/flan-t5-small"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

prompt = "Respond to: What's up!"
inputs = tokenizer(prompt, return_tensors="pt")
outputs = model.generate(**inputs, max_new_tokens=40)
response = tokenizer.decode(outputs[0], skip_special_tokens=True)

with SectionPrinter("Neural Generation"):
    print(response)


Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


=====================Neural Generation======================
- I'm a snob.


## Agent

In [ ]:
def agent_loop(user_message, tools, llm, max_steps=5):
    history = [{"role": "user", "content": user_message}]
    for _ in range(max_steps):
        response = llm(history, tools=tools)
        tool_call = response.get("tool_calls")
        if tool_call:
            tool_name = tool_call.get("name")
            args = tool_call.get("arguments")
            if not isinstance(tool_name, str) or tool_name not in tools:
                history.append({"role": "assistant", "tool_call": tool_call})
                history.append({"role": "tool", "name":str(tool_name), "content": f"error: unknown tool {tool_name!r}"}) 
                continue
            if not isinstance(args, dict):
                history.append({"role": "assistant", "tool_call": tool_call})
                history.append({"role": "tool", "name":str(tool_name), "content": f"error: invalid arguments {args!r}"})
                continue
            fn = tools[tool_name]
            result = fn(**args)
            history.append({"role": "assistant", "tool_call": tool_call})
            history.append({"role": "tool", "name":str(tool_name), "content": result})
        else:
            return response["content"]
    return "I could not complete that task in the step budget."

            

## Hybrid routing

In [ ]:
"""
def hybrid_chat(user_input):
    if is_destructive_action(user_input):
        return structured_flow(user_input)

    faq_answer = faq_respond(user_input, 0.6):
    if faq_answer:
        return faq_answer

    return agent_loop(user_input, tools, llm)

def is_destructive_action(user_input):
    danger_words = [
        "delete",
        "reset",
        "cancel",
        "quit",
        "exit",
        "stop",
        "terminate",
    ]
    return any(w in user_input.lower() for w in danger_words)
"""

Deterministic rules for anything destructive, retrieval for canned FAQs, LLM agent for everything else.